[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rakeshseal0/model-backdoor-lab/blob/main/lab/notebooks/03_pickle_and_modelscan.ipynb)

# 03 — Serialization: what a scanner can and cannot tell you

**Slot: 58–72 min. No GPU needed** — switch the runtime to CPU if you like.

Three artifacts, three verdicts:

| Artifact | Scanner says | Actually |
|---|---|---|
| `benign_model.pkl` | clean | clean |
| `attack_fixture.pkl` | **flagged** | runs code on load |
| your poisoned adapter (safetensors) | clean | **backdoored** |

That third row is the entire point.

> ⚠️ `attack_fixture.pkl` is a real malicious pickle. Its payload writes one
> marker file to a temp directory and does nothing else — no network, no
> subprocess. **You will disassemble it. You will not load it.**

> **The next cell prints a red `ERROR: pip's dependency resolver…` block.
> That is expected. Nothing is broken.**
>
> We install `modelaudit` with `--no-deps` on purpose. Its full dependency
> list pulls `gcsfs`, `s3fs`, `aiobotocore`, `google-cloud-storage` and
> `posthog` — cloud storage clients for scanning remote URLs, and an
> analytics client. This notebook scans local files and reports to nobody,
> so it needs none of them. Skipping them takes the install from **242
> seconds to about 5**, and the scan results are byte-identical.
>
> pip is telling you those packages are absent. They are absent
> deliberately. Installing fewer things is the security-conscious default,
> not a workaround.

In [ ]:
import os
# modelaudit sends usage analytics to a.promptfoo.app unless told not to.
# Set before install so nothing about your runtime is reported.
os.environ['PROMPTFOO_DISABLE_TELEMETRY'] = '1'
os.environ['NO_ANALYTICS'] = '1'

!pip -q install 'safetensors>=0.4.3'

# ModelScan pins itself to python<3.13; the cap is untested-version, not
# broken-code. Retry past it, and carry on without it if that fails too.
!pip -q install 'modelscan==0.8.*' \
  || pip -q install --ignore-requires-python 'modelscan==0.8.*' \
  || echo 'modelscan unavailable on this Python — labkit fallback scanner will be used'

# Second scanner, installed lean. A plain `pip install modelaudit` drags in
# gcsfs, s3fs, aiobotocore and google-cloud-* — 242 s measured on a cold
# runtime, for cloud-URL support this notebook never uses. --no-deps plus
# the three things it actually imports is 5 s and produces byte-identical
# findings on both fixtures. Colab already ships numpy/click/pydantic/rich.
!pip -q install --no-deps 'modelaudit>=0.2.50,<0.3' modelaudit-picklescan \
  && pip -q install yaspin cyclonedx-python-lib \
  || echo 'modelaudit unavailable — the comparison cells will say so and skip'

In [ ]:
# Pull labkit into the Colab runtime.
#
# Always re-clone rather than skipping when labkit/ exists. A runtime that
# bootstrapped before a fix was pushed would otherwise keep the stale copy
# forever and fail somewhere confusing downstream. The repo is small; this
# costs a second or two.
# Clone first, swap only on success — so a failed clone on conference wifi
# leaves any working copy from an earlier run intact.
import os, sys, pathlib, shutil
shutil.rmtree('_lab', ignore_errors=True)
!git clone -q https://github.com/rakeshseal0/model-backdoor-lab.git _lab

if pathlib.Path('_lab/lab/labkit').is_dir():
    shutil.rmtree('labkit', ignore_errors=True)
    shutil.copytree('_lab/lab/labkit', 'labkit')
elif not pathlib.Path('labkit').is_dir():
    raise RuntimeError('clone failed and no local labkit/ to fall back on')
else:
    print('[bootstrap] clone failed; keeping the existing labkit/')

sys.path.insert(0, '.')
# Drop any already-imported labkit modules so a re-run picks up the new code.
for _m in [_m for _m in list(sys.modules) if _m.startswith('labkit')]:
    del sys.modules[_m]

import labkit.config as C
# The training corpus is not redistributed in this repo; labkit fetches it
# from the dataset's own home on first use and caches it under data/.
print('trigger :', C.TRIGGER)
print('target  :', C.TARGET_MARKER)

### Step 1 — build the two fixtures

Building the malicious pickle is safe: `pickle.dump` calls `__reduce__` to
*describe* a function call, it does not perform it. The payload only runs
on **load**. That asymmetry is the vulnerability.

In [ ]:
from labkit.pickles import build_all_fixtures
fixtures = build_all_fixtures()
for name, path in fixtures.items():
    print(f'{name:<8} {path}  ({path.stat().st_size} bytes)')

### Step 2 — disassemble, don't load

`pickletools.dis` parses the opcode stream as data. Read the output and
find where it names a function to call.

In [ ]:
from labkit.pickles import disassemble
print(disassemble(fixtures['attack']))

#### ✏️ Fill in

| Question | Your answer |
|---|---|
| Which opcode names a function to import? | |
| What module and function does it name? | |
| Which opcode actually calls it? | |
| How many bytes is the whole file? | |

Now the same for the benign pickle. Note what is *absent*.

In [ ]:
print(disassemble(fixtures['benign']))

### Step 3 — what happens if you load it?

Don't. But try, so you see the guard.

In [ ]:
from labkit.pickles import load_fixture
try:
    load_fixture(fixtures['attack'])
except RuntimeError as e:
    print('REFUSED:', e)

### Step 4 — run a real scanner

ModelScan asks one question: *can loading this file execute code?*

If the cell above printed `modelscan unavailable on this Python`, don't
worry — `scan()` falls back to labkit's own opcode report and reaches the
same verdicts. The bracketed name in each line tells you which one ran.

In [ ]:
from labkit.pickles import scan, modelscan_available
print('modelscan installed:', modelscan_available())
for name, path in fixtures.items():
    r = scan(path)
    print(f"{name:<8} verdict={r['verdict']:<6} "
          f"findings={len(r.get('findings', []))}  [{r['scanner']}]")

### Step 4b — ask a second scanner the same question

One scanner teaches you to run the scanner. Two teach you that a scanner
is an *opinion* with a coverage boundary.

`modelaudit` walks the same opcode stream and reports more on the same
file — including a nested pickle payload ModelScan never mentions, and a
rule code for each finding that you can go and read.

In [ ]:
from labkit.pickles import audit, modelaudit_available
print('modelaudit installed:', modelaudit_available())
print()
for name, path in fixtures.items():
    a = audit(path)
    print(f"{name:<8} verdict={a['verdict']:<6} "
          f"findings={len(a['findings'])}")
    for f in a['findings']:
        rule = f['rule'] or '-'
        print(f"    {f['severity']:<9} {rule:<18} {f['message'][:52]}")

#### ✏️ Fill in

| | ModelScan | modelaudit |
|---|---|---|
| findings on `attack_fixture.pkl` | | |
| did either one *load* the file? | | |

Neither scanner unpickled anything. Both answers came from reading the
opcode stream.

### Step 5 — now scan the backdoored adapter

The adapter from notebook 01 is safetensors: a header plus raw tensor
bytes, with no opcode stream and no way to execute anything on load.

In [ ]:
from pathlib import Path
!git clone -q https://huggingface.co/{C.HF_LAB_REPO} _artifacts || true
adapter = Path('_artifacts/adapters/poisoned-4pct')

r = scan(adapter)
print(f"poisoned adapter: verdict={r['verdict']}")
print()
print('This adapter is backdoored. The scanner is not wrong —')
print('it answered the question it was asked.')

### Step 5b — the same adapter, the second scanner

Now run `modelaudit` over the adapter directory. It does **not** agree
with ModelScan. Before you read the next cell's output, predict which one
you think is right.

In [ ]:
a = audit(adapter)
print(f"poisoned adapter: modelscan={r['verdict']}  modelaudit={a['verdict']}")
print(f"modelaudit findings: {len(a['findings'])}")
print()
for f in a['findings']:
    where = f['file'] or '(directory as a whole)'
    print(f"  {f['severity']:<9} {f['message'][:56]:<58} in {where}")
print()
print('Look at the FILE column before you conclude anything.')

#### ✏️ Fill in — read the file column first

| Question | Your answer |
|---|---|
| How many findings did modelaudit report? | |
| Which file are the security findings in? | |
| How many are in `adapter_model.safetensors`? | |
| Did modelaudit detect the backdoor? | |

**Every security finding is in `README.md`** — the documentation *we* wrote
describing the attack. It matched the literal word "backdoor", an example
`requests.post` snippet, and an `AKIA…EXAMPLE` placeholder. Not one
finding touched a tensor.

Delete the README and the model is exactly as backdoored, and the scanner
goes quiet. That is a true result producing a false impression — and it
is the most useful thing in this notebook. A scanner matches patterns in
bytes. It does not understand your model.

#### ✏️ Fill in

| Artifact | ModelScan | modelaudit | Safe to load? | Safe to query? |
|---|---|---|---|---|
| `benign_model.pkl` | | | | |
| `attack_fixture.pkl` | | | | |
| poisoned adapter | | | | |

**safe to load ≠ safe to query.** Safetensors solved the first problem
completely. It was never trying to solve the second one — and neither
scanner was ever asked about it.